# Khaliq Lab - Intracellular Recordings of Substantia Nigra Dopaminergic Neurons

This notebook demonstrates how to access the NWB files published at [DANDI:001693](https://dandiarchive.org/dandiset/001693) ("Khaliq Embargo 2026").

**Embargo:** This dandiset is currently embargoed. Access requires a DANDI account that has been granted permission, and streaming requires authentication (shown below). Once the dandiset is public, the `client.dandi_authenticate()` call can be removed.

## Scientific context

The dataset contains whole-cell current-clamp recordings from dopaminergic neurons in the substantia nigra pars compacta, investigating how the axon initial segment (AIS) contributes to intrinsic firing properties. Each recording session (one subject on one date) contains multiple cells and three protocols per cell:

- **SF** - Spontaneous Firing (baseline intrinsic firing, no current injection)
- **CS** - Current Steps (response to stepwise current injection)
- **ADP** - After Depolarization (post-stimulus depolarization following brief current pulses)

Cells are classified as "with AIS component" or "without AIS component" to probe how AIS morphology shapes excitability.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import remfile
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

## Load a local session

While the dataset is being prepared for upload, we read directly from a local NWB file in the repository's `nwbfiles/` directory. The streaming code is preserved below (commented out) and can be re-enabled once the dandiset is published. With the current embargo, streaming requires `client.dandi_authenticate()`, which reads the DANDI token from the `DANDI_API_KEY` environment variable or prompts interactively.

In [ ]:
# Streaming from DANDI (dandiset is uploaded; auth still required while embargoed).
import h5py
import remfile
from dandi.dandiapi import DandiAPIClient

dandiset_id = "001693"
nwbfile_path = "sub-#3/sub-#3_ses-subject3++20211103_icephys.nwb"

with DandiAPIClient() as client:
    # Needed only while the dandiset is embargoed; remove once it is public.
    client.dandi_authenticate()
    asset = client.get_dandiset(dandiset_id, "draft").get_asset_by_path(nwbfile_path)
    # strip_query=False is important for embargoed assets: the signed query
    # string is what authorizes the S3 read. strip_query=True returns a bare
    # URL that 403s for non-public dandisets.
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=False)

rem_file = remfile.File(s3_url)
h5_file = h5py.File(rem_file, "r")
io = NWBHDF5IO(file=h5_file, load_namespaces=True)
nwbfile = io.read()

nwbfile

## Top-level session metadata

In [ ]:
print("session_description :", nwbfile.session_description)
print("session_start_time  :", nwbfile.session_start_time)
print("experimenter        :", nwbfile.experimenter)
print("lab                 :", nwbfile.lab)
print("institution         :", nwbfile.institution)
print()
print("experiment_description:")
print(nwbfile.experiment_description)

## Subject

Ages are stored as ISO 8601 durations (e.g. `P9.8Y` for 9.8 years). For subjects whose age was not recorded in the source spreadsheet (currently #4 and #6), we use the open-ended range `P0Y/` - meaning "0 years or older" - to represent total uncertainty without making an unverified claim about a lower bound. See `src/khaliq_lab_to_nwb/conversion_abf/conversion_notes.md` for details.

In [ ]:
nwbfile.subject

## Device and IntracellularElectrode

One device is shared across all recordings; each cell has its own `IntracellularElectrode` entry.

In [ ]:
device = nwbfile.devices["MultiClamp700B"]
device

In [ ]:
nwbfile.icephys_electrodes["IntracellularElectrodeCell12"]

## Experimental protocols

Each session ran three sequential current-clamp protocols on every cell:

- **SF (Spontaneous Firing)** - baseline intrinsic firing with no current injection. The cell runs free; we record its natural pacemaking pattern. Typically a single continuous sweep, no stimulus series in NWB.
- **CS (Current Steps)** - stepwise current injection of varying amplitudes. Each step is one sweep (~4 s long here), separated by a few seconds of recovery before the next step. Tests excitability: f-I relationship, rheobase, gain.
- **ADP (After Depolarization)** - brief depolarizing pulses delivered after holding at a hyperpolarized potential. Probes rebound depolarization and post-stimulus membrane dynamics.

Cells are partitioned into "with AIS component" vs "without AIS component" - the experimental condition the dataset compares.

### Protocol epochs

`nwbfile.epochs` is a `TimeIntervals` table with **one row per `(cell, protocol)` pair**. Each row spans the full window of that protocol on that cell - from the first sweep's start to the last sweep's end, inter-sweep gaps included. Custom columns:

- `protocol` - abbreviation (SF / CS / ADP)
- `cell_id` - the cell this protocol was run on
- `ais_status` - "with AIS component" / "without AIS component"
- `tags` - descriptive labels for the protocol type


In [ ]:
nwbfile.epochs.to_dataframe()

## Acquisition and stimulus: how the data is stored

The recordings are split between two NWB containers:

- `nwbfile.acquisition` - what was *measured*: voltage traces (`CurrentClampSeries`)
- `nwbfile.stimulus` - what was *commanded*: current waveforms (`CurrentClampStimulusSeries`)

For each `(cell, protocol)` combination, all sweeps from that protocol are concatenated into **one** `CurrentClampSeries` (voltage, in volts) and **one** `CurrentClampStimulusSeries` (current, in amperes). SF recordings have no stimulus series because no current is injected.

Both series carry an explicit per-sample `timestamps` array (in seconds from session start). Within a sweep, sampling is regular at 50 kHz; the gaps *between* sweeps - typically a few seconds while the experimenter advances the protocol - are real wall-clock pauses that the timestamps preserve. The stimulus series doesn't store its own timestamps array; it soft-links to the response's, so the timestamps dataset is only on disk once.

You can read either series directly:

```python
v_series   = nwbfile.acquisition["CurrentClampSeriesAisProtocolCSCell12"]
voltage    = v_series.data[:]          # all samples, in volts
timestamps = v_series.timestamps[:]    # absolute time of each sample, in seconds
```

Below we list the series in the file, then plot one full `(cell, protocol)` pair so the concatenation and inter-sweep gaps are visible.


In [ ]:
print(
    f"Acquisition (voltage) series : {len(nwbfile.acquisition)}  # one per (cell, protocol)"
)
print(
    f"Stimulus    (current) series : {len(nwbfile.stimulus)}     # SF has no stimulus"
)
print()
print("Acquisition series:")
for name in nwbfile.acquisition:
    print(f"  {name}")
print()
print("Stimulus series:")
for name in nwbfile.stimulus:
    print(f"  {name}")

### Visualize one full (cell, protocol) series per protocol

Below we plot the full concatenated trace for each of the three protocols on this cell. In every case the voltage axis is clipped to the physiological range (-120 to 60 mV) so saturation artifacts (flagged in `invalid_times`, shown later) don't blow up the y-axis. Flat segments between blocks of activity are real wall-clock waiting periods preserved by the timestamps.


In [ ]:
# CS - current steps. Has both voltage response and stimulus current command.
cs_response_name = next(n for n in nwbfile.acquisition if "ProtocolCS" in n)
cs_stimulus_name = cs_response_name.replace(
    "CurrentClampSeries", "CurrentClampStimulusSeries"
)

v_series = nwbfile.acquisition[cs_response_name]
i_series = nwbfile.stimulus[cs_stimulus_name]

t = np.asarray(v_series.timestamps[:])
v = np.asarray(v_series.data[:]) * v_series.conversion
i = np.asarray(i_series.data[:]) * i_series.conversion

fig, (ax_v, ax_i) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax_v.plot(t, v * 1e3, color="C0", linewidth=0.4)
ax_v.set_ylabel("Vm (mV)")
ax_v.set_ylim(-120, 60)
ax_v.set_title(f"{v_series.name}  (full concatenated series, {len(t):,} samples)")
ax_i.plot(t, i * 1e12, color="C1", linewidth=0.4)
ax_i.set_ylabel("I (pA)")
ax_i.set_xlabel("Time (s, from session start)")
plt.tight_layout()
plt.show()

#### SF - spontaneous firing

No current injection (`I = 0`), so there is no `CurrentClampStimulusSeries` and we only plot the voltage. This trace shows the cell's autonomous pacemaking - the regular train of action potentials it generates with no external drive.


In [ ]:
# SF - spontaneous firing. No stimulus series; plot voltage only.
sf_response_name = next(n for n in nwbfile.acquisition if "ProtocolSF" in n)
v_series = nwbfile.acquisition[sf_response_name]

t = np.asarray(v_series.timestamps[:])
v = np.asarray(v_series.data[:]) * v_series.conversion

fig, ax_v = plt.subplots(figsize=(11, 3), constrained_layout=True)
ax_v.plot(t, v * 1e3, color="C0", linewidth=0.4)
ax_v.set_ylabel("Vm (mV)")
ax_v.set_ylim(-120, 60)
ax_v.set_xlabel("Time (s, from session start)")
ax_v.set_title(f"{v_series.name}  (full series, {len(t):,} samples)")
plt.show()

#### ADP - after-depolarization

Each sweep starts with a hyperpolarizing pre-conditioning pulse, then the cell is released (or briefly depolarized) and the post-stimulus dynamics are recorded. Across sweeps, the holding voltage is varied. Both voltage response and the injected current command are present.


In [ ]:
# ADP - after-depolarization. Has both response and stimulus.
adp_response_name = next(n for n in nwbfile.acquisition if "ProtocolADP" in n)
adp_stimulus_name = adp_response_name.replace(
    "CurrentClampSeries", "CurrentClampStimulusSeries"
)

v_series = nwbfile.acquisition[adp_response_name]
i_series = nwbfile.stimulus[adp_stimulus_name]

t = np.asarray(v_series.timestamps[:])
v = np.asarray(v_series.data[:]) * v_series.conversion
i = np.asarray(i_series.data[:]) * i_series.conversion

fig, (ax_v, ax_i) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax_v.plot(t, v * 1e3, color="C0", linewidth=0.4)
ax_v.set_ylabel("Vm (mV)")
ax_v.set_ylim(-120, 60)
ax_v.set_title(f"{v_series.name}  (full concatenated series, {len(t):,} samples)")
ax_i.plot(t, i * 1e12, color="C1", linewidth=0.4)
ax_i.set_ylabel("I (pA)")
ax_i.set_xlabel("Time (s, from session start)")
plt.tight_layout()
plt.show()

## IntracellularRecordings table and sweep extraction

`nwbfile.intracellular_recordings` is the **per-sweep index** for the file. It's an `AlignedDynamicTable` that joins three sub-tables row by row:

| sub-table | column | what it holds |
|---|---|---|
| `electrodes` | `electrode` | reference to the `IntracellularElectrode` (which cell was being recorded) |
| `stimuli`    | `stimulus` | a `(start_index, count, series)` triple pointing into the stimulus series. `(None, None, None)` for SF sweeps with no stimulus |
| `responses`  | `response` | the analogous triple pointing into the response series |

**One row of this table is one sweep.** The `(start_index, count, series)` triple is NWB's `TimeSeriesReference` - a typed pointer saying "samples `[start_index, start_index + count)` of `series` are this sweep."


In [ ]:
nwbfile.intracellular_recordings.to_dataframe().head()

### Extract one sweep

To pull a single sweep, look up its IR row and slice the underlying series with the triple. Below we filter the table to CS sweeps and plot the middle one.


In [ ]:
# Walk the IR table directly: each row's responses.response is a (start_index, count, series) triple.
# Filter to rows whose response series is a CS series and pick a sweep in the middle.
ir_df = nwbfile.intracellular_recordings.to_dataframe()
response_col = ir_df[("responses", "response")]
cs_mask = response_col.apply(lambda ref: "ProtocolCS" in ref[2].name)
cs_rows = ir_df[cs_mask]

sweep_pos = len(cs_rows) // 2
middle = cs_rows.iloc[sweep_pos]
v_start, v_count, v_series = middle[("responses", "response")]
i_start, i_count, i_series = middle[("stimuli", "stimulus")]

# Slice the concatenated series to extract just this sweep
voltage = np.asarray(v_series.data[v_start : v_start + v_count])
current = np.asarray(i_series.data[i_start : i_start + i_count])
t_v = np.asarray(v_series.timestamps[v_start : v_start + v_count])
t_i = np.asarray(i_series.timestamps[i_start : i_start + i_count])

fig, (ax_v, ax_i) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax_v.plot(t_v, voltage * 1e3, color="C0", linewidth=0.7)
ax_v.set_ylabel("Vm (mV)")
ax_v.set_title(f"{v_series.name}  (sweep {sweep_pos + 1} of {len(cs_rows)})")
ax_i.plot(t_i, current * 1e12, color="C1", linewidth=0.7)
ax_i.set_ylabel("I (pA)")
ax_i.set_xlabel("Time (s, from session start)")
plt.tight_layout()
plt.show()

## Invalid times - voltage saturation artifacts

`nwbfile.invalid_times` flags intervals where the recorded voltage exceeded the amplifier's usable range (detected as absolute voltage beyond 200 mV). These samples do not reflect true membrane potential and should be excluded from analysis.

Custom columns:

- `time_series` - reference to the affected `CurrentClampSeries`
- `time_series_name` - the name of that series (convenient for grouping)
- `voltage_min`, `voltage_max` - extremes observed inside the interval (V)

In [ ]:
invalid_df = nwbfile.invalid_times.to_dataframe()
invalid_df[
    ["start_time", "stop_time", "time_series_name", "voltage_min", "voltage_max"]
]

### Visualizing a saturation interval

We overlay the invalid interval on the affected voltage trace to make the artifact visible.

In [ ]:
row = invalid_df.iloc[0]
affected_name = row["time_series_name"]
affected = nwbfile.acquisition[affected_name]

# Acquisition series store explicit timestamps (one per sample).
t = np.asarray(affected.timestamps[:])
v = np.asarray(affected.data[:]) * affected.conversion

# Zoom in around the saturation interval so the artifact is legible
window_pad = 2.0  # seconds on each side
start, stop = float(row["start_time"]), float(row["stop_time"])
mask = (t >= start - window_pad) & (t <= stop + window_pad)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t[mask], v[mask] * 1e3, color="C0", linewidth=0.7, label="Vm")
ax.axvspan(start, stop, color="red", alpha=0.3, label="invalid_times")
ax.set_xlabel("Time (s, from session start)")
ax.set_ylabel("Vm (mV)")
ax.set_title(f"{affected_name} - saturation flagged in red")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

## Icephys hierarchical tables

NWB's icephys extension organizes intracellular recordings into a four-level hierarchy. Each higher level groups rows from the level below. We've already seen the bottom level (`intracellular_recordings`); the higher levels are descriptive metadata that lets analyses iterate at the right granularity:

1. `intracellular_recordings` - one row per sweep (shown above)
2. `icephys_simultaneous_recordings` - groups sweeps recorded at the same instant. One row per sweep here, since recordings are sequential per electrode.
3. `icephys_sequential_recordings` - groups simultaneous recordings sharing a stimulus protocol. **One row per (cell, protocol).** Custom `protocol` column names the protocol abbreviation (SF/CS/ADP).
4. `icephys_repetitions` - groups sequential recordings that are repetitions of the same experimental run. **One row per cell.** Custom columns: `cell_id`, `ais_status`, `electrode_name`.
5. `icephys_experimental_conditions` - groups repetitions by experimental condition (here, AIS status). Custom column `ais_component_status`.


In [ ]:
nwbfile.icephys_simultaneous_recordings.to_dataframe().head()

In [ ]:
# One row per (cell, protocol). The custom `protocol` column shows SF/CS/ADP.
nwbfile.icephys_sequential_recordings.to_dataframe()

In [ ]:
# One row per cell. Custom columns: cell_id, ais_status, electrode_name.
nwbfile.icephys_repetitions.to_dataframe()

In [ ]:
# One row per experimental condition (AIS status).
nwbfile.icephys_experimental_conditions.to_dataframe()